# Redrob Hackathon — Candidate Ranking Pipeline

**Goal:** Rank the top 100 candidates from 100,000 for a Senior AI Engineer role.

**Runtime required:** GPU (T4 or better) — use *Runtime → Change runtime type → T4 GPU*

---

### Steps
1. Clone repo + install dependencies
2. Upload `candidates.jsonl`
3. Pre-compute embeddings (GPU, ~20–40 min)
4. Rank candidates (CPU, <2 min)
5. Validate + download `submission.csv`

## Step 1 — Setup

Clone the repo and install all dependencies.

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
print(result.stdout.split('\n')[0])
print("GPU OK")

In [ ]:
# Clone the repo
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs

In [ ]:
# Install dependencies
# torch + torchvision are pre-installed in Colab and CUDA-optimized.
# Reinstalling them (from requirements.txt) breaks the torchvision CUDA build.
# We only install the packages Colab doesn't already provide.
!pip install -q sentence-transformers tqdm pandas pyarrow
print("Dependencies installed.")

## Step 2 — Upload Data

Upload `candidates.jsonl` (465 MB). Choose **one** of the two options below.

In [ ]:
# Option A: Upload directly from your computer
# (browser file picker — works for files up to ~1 GB)
import shutil
from google.colab import files

print("Select candidates.jsonl from your computer...")
uploaded = files.upload()

# Move to project root
for fname in uploaded:
    shutil.move(fname, 'candidates.jsonl')
print("Uploaded:", list(uploaded.keys()))

In [ ]:
# Option B: Load from Google Drive (skip if you used Option A)
# Upload candidates.jsonl to your Drive first, then set the path below.

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/candidates.jsonl', 'candidates.jsonl')
# print("Copied from Drive.")

In [ ]:
# Verify the data file is present
import os
size_mb = os.path.getsize('candidates.jsonl') / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400, "File looks too small — check the upload"

## Step 3 — Pre-computation (GPU)

Embeds all 100K candidate profiles using `BAAI/bge-large-en-v1.5` and extracts structured features.

**Expected time:** ~20–40 min on T4, ~10–15 min on A100.

Outputs saved to `./artifacts/`: `embeddings.npy` (~400 MB), `jd_embedding.npy`, `features.parquet`.

In [ ]:
!python src/precompute.py \
    --candidates candidates.jsonl \
    --artifacts  ./artifacts

In [ ]:
# Verify artifacts
import numpy as np
import pandas as pd

emb = np.load('artifacts/embeddings.npy')
jd  = np.load('artifacts/jd_embedding.npy')
df  = pd.read_parquet('artifacts/features.parquet')

assert emb.shape == (100000, 1024), f"Unexpected shape: {emb.shape}"
assert jd.shape  == (1024,),        f"Unexpected shape: {jd.shape}"
assert len(df)   == 100000,          f"Unexpected row count: {len(df)}"

print(f"embeddings.npy : {emb.shape}  ({emb.nbytes/1e6:.0f} MB)")
print(f"jd_embedding   : {jd.shape}")
print(f"features.parquet: {len(df)} candidates, {len(df.columns)} columns")
print("Artifacts OK")

## Step 4 — Rank Candidates (CPU)

Scores all 100K candidates using the multi-factor formula and selects the top 100.

**Expected time:** <2 min.

In [ ]:
!python src/rank.py \
    --artifacts ./artifacts \
    --out       ./jashwanth_s.csv

## Step 5 — Validate & Download

Run the official validator, inspect the top 10, then download `submission.csv`.

In [ ]:
# Quick sanity checks
df = pd.read_csv('jashwanth_s.csv')

assert len(df) == 100,                              "Must have exactly 100 rows"
assert set(df['rank']) == set(range(1, 101)),       "Ranks must be 1-100 exactly once"
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), "Scores must be non-increasing"
assert df['candidate_id'].str.match(r'^CAND_\d{7}$').all(), "Unexpected candidate_id format"

# Honeypot check
import sys
sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
from dataclasses import fields

feat_df = pd.read_parquet('artifacts/features.parquet')
top100_ids = set(df['candidate_id'])
top100_feats = feat_df[feat_df['candidate_id'].isin(top100_ids)]

honeypots = []
for _, row in top100_feats.iterrows():
    kwargs = {f.name: row[f.name] for f in fields(CandidateFeatures) if f.name in row.index}
    kwargs.setdefault('profile_text', '')
    if is_honeypot(CandidateFeatures(**kwargs)):
        honeypots.append(row['candidate_id'])

print(f"Rows         : {len(df)}")
print(f"Score range  : {scores[0]:.4f} → {scores[-1]:.4f}")
print(f"Honeypots    : {len(honeypots)} (must be 0)")
assert len(honeypots) == 0, f"Honeypots in top 100: {honeypots}"
print("All checks passed.")

In [ ]:
# Inspect top 10
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Download jashwanth_s.csv to your computer
from google.colab import files
files.download('jashwanth_s.csv')
print("jashwanth_s.csv downloaded.")